# Clinical baseline and decision-curve analysis

## Phase 0 · Predictor inventory

Run these cells on the eight-week SSI export in the authorized environment.
The three output files contain predictor metadata. Use them to confirm the
clinical feature mapping before fitting the clinical baselines or making
decision curves.

In [ ]:
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../..').resolve()))
from ssi.feature_filter import feature_filter
from ssi.preprocess import preprocess

DATA_PATH = Path('../ssi_all_features.parquet')
results_dir = Path('results/clinical_baseline_inventory')
excluded = {'birthid', 'mother_patid', 'target_variable'}

In [ ]:
df = pd.read_parquet(DATA_PATH)
raw_catalog = (
    df.loc[~df['feature_name'].isin(excluded) & df['description'].ne('phenotype'),
           ['feature_name', 'description']]
    .drop_duplicates()
    .sort_values(['feature_name', 'description'])
)
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
with redirect_stdout(io.StringIO()):
    prepared = preprocess(df_pivoted, defer_imputation=True)
    predictors = prepared.drop(columns=list(excluded))
    available = sorted(feature_filter(predictors, skip_medadmin_ingredient=True).columns)
if not available or len(available) != len(set(available)):
    raise ValueError('Expected a nonempty list of unique predictor names.')

In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
raw_catalog.to_csv(results_dir / 'clinical_baseline_raw_feature_catalog.csv', index=False)
pd.Series(available, name='feature_name').to_csv(
    results_dir / 'clinical_baseline_available_predictor_columns.csv', index=False,
)

retained = raw_catalog[raw_catalog['feature_name'].isin(available)]
groups = retained.groupby('description')['feature_name'].nunique().sort_index()
# These are name matches for review. Clinical interpretation requires source definitions.
family_terms = {
    'Obesity': ('bmi', 'obes', 'e66', 'o99_21', 'wt_'),
    'Diabetes': ('diabet', 'o24', 'e11', 'glucose', 'a1c'),
    'Hypertensive disorders': ('hypertens', 'preeclamp', 'o13', 'o14', 'systolic'),
    'Prior cesarean': ('prior_cesarean', 'o34_21', 'z98_891'),
    'Membrane rupture / labor': ('o42', 'membrane_ruptur', 'stage_labor'),
    'Chorioamnionitis': ('chorio', 'o41_12'),
    'Estimated blood loss': ('blood_loss', 'ebl'),
    'Antibiotic timing': ('2180', 'cefazolin', 'prophylax'),
    'Temperature': ('temperature', 'body_temp', '8310-5'),
    'WBC': ('wbc', 'leukocyte', '6690-2', '26464-8', '5821-4'),
}
summary = [
    f'Raw predictor names: {raw_catalog["feature_name"].nunique()}',
    f'Available post-filter predictors: {len(available)}',
    '',
    'Predictor counts by source description (mapped names):',
    *(f'{group}: {count}' for group, count in groups.items()),
    f'Generated or unmapped features: {len(set(available) - set(retained["feature_name"]))}',
    '',
    'Candidate families by name (source and timing verification pending):',
]
for group, terms in family_terms.items():
    matches = sum(any(term in name.lower() for term in terms) for name in available)
    summary.append(f'{group}: {matches} name matches')
(results_dir / 'clinical_baseline_inventory_summary.txt').write_text(
    '\n'.join(summary) + '\n', encoding='utf-8',
)
print(f'Wrote {len(available)} predictor names and source metadata to {results_dir}')

## Next step · Review and freeze the clinical feature mapping

Review the exported names and verify their source and time window with the
clinical collaborators. Record the approved columns and rationale before
running the clinical LR and RF comparisons and choosing thresholds for
decision-curve analysis.